# 📘 FABRIC JupyterHub Access & Setup
## Scientific Workflows for Brain and Behavioral Research
### National Science Foundation Supported Learning Initiative (Award No. OAC-2417875)

> ## ⚠️ Optional Advanced Resource
>
> **This notebook is not required for the workshop.** It is the natural next step after **Pre-Workshop Notebook 3**, for participants who have already created a FABRIC account, been approved on the workshop's FABRIC project, and want to provision and run code the way FABRIC itself recommends.
>
> **This notebook does not run in Google Colab.** It runs only inside **FABRIC's own JupyterHub**, where `fablib` (FABRIC's Python toolkit) is already installed and your credentials are already configured — no `pip install`, no manually uploading tokens. Section 1 below explains exactly how to get this notebook open there.

***

## 📋 Set Up Your Profile
*Run the cell below to record your name and institution in this notebook.*


In [ ]:
# PROFILE SETUP
student_name = "Your Name"
institution = "Your University/Organization"
research_area = "Neuroscience / Psychology / Biology / Engineering / Computer Science / Other"

print(f"✅ Notebook initialized for {student_name} ({institution})")
print(f"Research area: {research_area}")

***
## ⚠️ Before You Begin — Checklist

You should be able to check off all of these before continuing. If you can't, go back to **Pre-Workshop Notebook 3, Step 1** first.

1. **You have a FABRIC account** and have accepted the portal's cookie policy.
2. **You have been approved** on the workshop's FABRIC project by your Project Lead (contact your organizer if you're unsure).
3. **You have logged into FABRIC's JupyterHub at least once**, and run FABRIC's own official **"Configure your Jupyter Environment"** notebook (it appears automatically the first time you open JupyterHub, via the `start_here.ipynb` index). This is what sets up your credentials and SSH keys automatically — everything below assumes it has already run successfully.
4. **Your code is in a GitHub repository** you can `git clone` (public, or one you have clone access to). If your code currently only lives in Google Drive, push it to a GitHub repo first — Drive mounting isn't available inside FABRIC's JupyterHub the way it is in Colab.

***
## 1 · 📂 How to Open This Notebook Inside FABRIC's JupyterHub

Since this notebook can't run in Colab, get it into JupyterHub one of these ways:

**Option A — clone the workshop repo (recommended).** Open a Terminal from JupyterHub's Launcher (File → New → Terminal), then run:
```
git clone https://github.com/cyneuro/CI-Workshop-Colabs.git
```
Then use the file browser on the left to navigate into `CI-Workshop-Colabs/` and open `Capstone_Access_FABRIC_JupyterHub.ipynb`.

**Option B — download just this file.** From the JupyterHub file browser, use File → Open From URL and paste the raw GitHub URL for this notebook, or download it from the workshop website and use the file browser's **Upload** button.

***


## 2 · ✅ Confirm Your Environment Is Ready 🪐

Unlike the Colab version of this workshop step, there's no installing or token-path setup here — FABRIC's JupyterHub already did that when you ran its "Configure your Jupyter Environment" notebook. This cell just confirms it worked.

In [ ]:
# 🪐 Confirm fablib is configured and can see your account
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

try:
    fablib = fablib_manager()
    fablib.show_config()
    print("\n✅ Connected! fablib recognizes your credentials.")
except Exception as e:
    msg = str(e)
    print(f"⚠️ Something isn't configured yet: {msg}")
    print("   Go back to JupyterHub's Launcher and re-run the official 'Configure your Jupyter")
    print("   Environment' notebook (from start_here.ipynb) — that sets up the credentials this")
    print("   cell depends on. If it still fails after that, your token may have expired; the")
    print("   Configure notebook also handles renewing it.")


***
## 3 · 📂 Point to Your Code on GitHub

Fill in your repository URL and the script you want run once your computing session is ready.

In [ ]:
# 📂 Your code source
github_repo_url = "https://github.com/your-username/your-repo.git"  #@param {type:"string"}
entry_script     = "run.py"  #@param {type:"string"}

print(f"✅ Will clone: {github_repo_url}")
print(f"✅ Entry script to run on the computing session: {entry_script}")


***
## 4 · 🪐 Request Your Computing Session

Modest, workshop-appropriate defaults — change these if you know you need more.

In [ ]:
# 🪐 Define and submit a one-node computing session
SLICE_NAME = f"Workshop-{student_name.replace(' ', '')}"  #@param {type:"string"}
CORES = 2      #@param {type:"integer"}
RAM_GB = 8     #@param {type:"integer"}
DISK_GB = 10   #@param {type:"integer"}

try:
    slice = fablib.new_slice(name=SLICE_NAME)
    site = fablib.get_random_site()
    slice.add_node(name="Node1", site=site, cores=CORES, ram=RAM_GB, disk=DISK_GB, image="default_ubuntu_22")
    slice.submit()
    print(f"✅ Requested computing session '{SLICE_NAME}' at site {site}. Continue to Section 5 to wait for it to become Ready.")
except Exception as e:
    print(f"⚠️ Request failed: {e}")
    print("   Common causes: the chosen site is temporarily full, or your project quota is exhausted.")
    print("   Re-run this cell to try a different (randomly chosen) site, or contact your organizer.")


***
## 5 · ⏳ Wait Until Your Session Is Ready

Checks every 15 seconds until the session reaches **Ready**.

In [ ]:
# 🪐 Poll until the session is Ready (or report a problem)
import time

TIMEOUT_SECONDS = 600
waited = 0

try:
    slice = fablib.get_slice(name=SLICE_NAME)
    while waited < TIMEOUT_SECONDS:
        state = slice.get_state()
        print(f"⏳ [{waited}s] Current state: {state}")
        if state == "StableOK":
            print("✅ Ready! Continue to Section 6.")
            break
        if state == "StableError":
            print("⚠️ Not Ready — some part of the request failed. See the state table in Notebook 3's")
            print("   Reference Guide, then try Section 4 again, or contact your organizer.")
            break
        time.sleep(15)
        waited += 15
    else:
        print("⚠️ Timed out waiting. Re-run this cell to keep checking — FABRIC is likely just busy.")
except Exception as e:
    print(f"⚠️ Could not check status: {e}")


***
## 6 · ▶️ Clone and Run Your Code

This clones your GitHub repository directly onto the computing session (over FABRIC's own network — no upload step needed) and runs your entry script.

In [ ]:
# 🪐 Clone your repo on the node and run it
try:
    slice = fablib.get_slice(name=SLICE_NAME)
    node = slice.get_node(name="Node1")

    stdout, stderr = node.execute(f"git clone {github_repo_url} /home/ubuntu/my_code")
    print(f"✅ Cloned {github_repo_url} on the computing session.")

    print(f"\n▶️ Running {entry_script} ...\n")
    stdout, stderr = node.execute(f"cd /home/ubuntu/my_code && python3 {entry_script}")
    print("----- Output -----")
    print(stdout)
    if stderr:
        print("----- Errors (if any) -----")
        print(stderr)
except Exception as e:
    print(f"⚠️ Something went wrong cloning or running your code: {e}")
    print("   Double-check the repository URL is correct and public (or that your key has clone")
    print("   access), and that the entry script filename matches exactly.")


***
## 7 · 📥 Retrieve Your Results

Brings any output files your script wrote back onto this notebook's own storage. Use JupyterHub's file browser afterward to download a copy to your own computer — treat the computing session itself as temporary (Concept 11 in Notebook 3's Reference Guide).

In [ ]:
# 🪐 Download results from the computing session
RESULTS_REMOTE_PATH = "/home/ubuntu/my_code/results"  #@param {type:"string"}
RESULTS_LOCAL_PATH   = "./fabric_results"                #@param {type:"string"}

import os
os.makedirs(RESULTS_LOCAL_PATH, exist_ok=True)

try:
    node = fablib.get_slice(name=SLICE_NAME).get_node(name="Node1")
    node.download_directory(RESULTS_LOCAL_PATH, RESULTS_REMOTE_PATH)
    print(f"✅ Downloaded results to {RESULTS_LOCAL_PATH} (visible in the file browser on the left).")
    print("   Download a copy to your own computer from there — don't rely on this being backed up.")
except Exception as e:
    print(f"ℹ️ Nothing downloaded ({e}). If your script didn't write files to {RESULTS_REMOTE_PATH}, this is expected.")


***
## 8 · 🪐 Release Your Computing Session

> ⚠️ **This step is required, not optional.** FABRIC is a shared national resource — leaving a session running after you're done takes resources away from other researchers, including other workshop participants.

In [ ]:
# 🪐 Release resources back to the shared pool (REQUIRED)
try:
    fablib.get_slice(name=SLICE_NAME).delete()
    print(f"✅ Released '{SLICE_NAME}'. Thank you for cleaning up after yourself!")
except Exception as e:
    print(f"⚠️ Could not release automatically: {e}")
    print("   Log into the FABRIC Portal and delete the slice manually to be safe.")


In [ ]:
# 🪐 Confirm the session is really gone
try:
    fablib.get_slice(name=SLICE_NAME)
    print("⚠️ The session still appears to exist — re-run the cell above, or release it from the FABRIC Portal.")
except Exception:
    print(f"✅ Confirmed: '{SLICE_NAME}' no longer exists. You're all cleaned up.")


### ✍️ Reflection
*Double-click this cell to write your response.*

* **What I observed:** Compare this walkthrough to Notebook 3's Colab version — what got simpler once you were working inside FABRIC's own JupyterHub instead of Colab?
* **Connecting to key concepts:** Why does FABRIC provide its own JupyterHub environment (rather than expecting researchers to configure `fablib` from an arbitrary external machine every time)? What does that trade off, and for whom?


***
## 📖 Reference Guide: Key Concepts for This Guide

*This builds on Notebook 3's 15-concept Reference Guide — see that notebook for the fundamentals (research testbed, computing session, secure gateway, session states, community resource ethics, and so on). The concepts below are specific to working inside FABRIC's own JupyterHub.*

1. **Configure Your Jupyter Environment notebook** – FABRIC's own official first-run notebook (reached via `start_here.ipynb`) that sets up your credentials, bastion key, and slice key automatically the first time you use JupyterHub — the step this guide assumes is already done.
2. **Native Network Path** – Because FABRIC's JupyterHub sits inside FABRIC's own infrastructure, connections to the secure gateway and to computing nodes don't have to cross the public internet the way they would from an external machine like a Colab VM — this is part of why it's more reliable.
3. **Token Auto-Refresh** – FABRIC identity tokens expire (by default after a few hours); JupyterHub's Configure notebook can renew yours without repeating a full browser sign-in every time, unlike a one-off external environment.
4. **Persistent Hub Storage** – Your JupyterHub home directory persists between logins, unlike a Colab session's disk — but it is still not a substitute for backing up important results to your own computer.
5. **`git clone` Over FABRIC's Network** – Cloning your code directly on the computing node (Section 6) avoids the upload step Colab needed, since the node has its own internet access once it's Ready.


***
## 🔗 External Resources & Further Study

### Official FABRIC JupyterHub Getting Started
| Resource | Link |
|----------|------|
| `jupyter-examples` repo (start here) | https://github.com/fabric-testbed/jupyter-examples |
| `start_here.ipynb` (index notebook) | https://github.com/fabric-testbed/jupyter-examples/blob/main/start_here.ipynb |
| teaching-materials: Getting Started | https://github.com/fabric-testbed/teaching-materials/blob/main/Getting%20Started.md |
| fablib installation & configuration docs | https://fabric-fablib.readthedocs.io/ |
| "Hello, FABRIC" full walkthrough tutorial | https://teaching-on-testbeds.github.io/hello-fabric/ |

### From Notebook 3 (still relevant here)
| Resource | Link |
|----------|------|
| FABRIC Portal | https://portal.fabric-testbed.net/ |
| FABRIC Documentation | https://learn.fabric-testbed.net/ |
| FABRIC Project Teams sheet | https://docs.google.com/spreadsheets/d/1zwl_Fwshny4wWIfUFN6q38u-Nar0ZSu5/edit?gid=1361837947#gid=1361837947 |

***
*End of FABRIC JupyterHub Access & Setup*